In [1]:
# Uncomment line below to install exlib
# !pip install diskcache
import sys; 

ROOT_DIR = '../..'
sys.path.append(f'{ROOT_DIR}/src')



import openai
import os

# with open(f"{ROOT_DIR}/API_KEY.txt", "r") as file:
#     api_key = file.read().strip()
# with open(f"{ROOT_DIR}/API_KEY.txt", "r") as file:
#     api_key = file.read().strip()
import json
with open(f"{ROOT_DIR}/API_KEYS2.json", "r") as file:
    api_keys = json.load(file)

os.environ['OPENAI_API_KEY'] = api_keys['OPENAI_API_KEY']
os.environ['ANTHROPIC_API_KEY'] = api_keys['ANTHROPIC_API_KEY']
os.environ['GOOGLE_API_KEY'] = api_keys['GOOGLE_API_KEY']
os.environ['CACHE_DIR'] = os.path.join(ROOT_DIR, 'cache_dir3')

# Cholec

In [2]:
import importlib
import sys; sys.path.append("../src")
import cholec
importlib.reload(cholec)
from cholec import get_llm_generated_answer
from cholec import CholecExample, CholecDataset, load_model, items_to_examples
from cholec import isolate_individual_features, distill_relevant_features, calculate_expert_alignment_scores

In [3]:
test_dataset = CholecDataset(split="test")

In [4]:
from tqdm.auto import tqdm
import json

In [5]:
# model = 'gpt-4o'
models = [
    'gpt-4o',
    'claude-3-5-sonnet-latest',
    'gemini-2.0-flash',
    'o1'
]

eval_model_name = 'gemini-2.0-flash'
eval_model = load_model(eval_model_name)


In [6]:
methods = [
    'vanilla', 
    # 'cot', 
    # 'socratic', 
    # 'subq'
]

In [7]:
import torch
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [8]:
import json
import copy
from tqdm.auto import tqdm

# for model in models:
#     print(f"=== Using model {model} ===")
#     for method in methods:
        # print(f"=== Using method {method} ===")

model = models[0]
method = methods[0]
    
load_path = os.path.join(ROOT_DIR, f'results/{method}/cholec_{model}.json')
save_path = os.path.join(ROOT_DIR, f'results/{method}/cholec_{model}_{eval_model_name}.json')

with open(load_path) as input_file:
    results = json.load(input_file)
    
results[0].keys()

dict_keys(['id', 'true_safe_list', 'true_unsafe_list', 'llm_raw_output', 'llm_explanation', 'llm_safe_list', 'llm_unsafe_list', 'all_claims', 'relevant_claims', 'alignable_claims', 'aligned_category_ids', 'alignment_scores', 'alignment_reasonings', 'final_alignment_score', 'safe_iou', 'unsafe_iou', 'alignment_categories', 'alignment_raws'])

In [9]:
results[0]['id'], results[1]['id'], results[2]['id']

('cholec80_video52_008.png',
 'M2CCAI2016_video95_004.png',
 'M2CCAI2016_video107_007.png')

In [10]:
test_dataset.__dict__.keys()

dict_keys(['dataset', 'image_size', 'preprocess_image', 'preprocess_labels'])

In [11]:
id2idx_mapping = {
    test_dataset.dataset[i]['id']: i
    for i in range(len(test_dataset.dataset))
}

In [12]:
test_dataset[id2idx_mapping[results[0]['id']]]['image'];

In [13]:
test_dataset.dataset[0]['id']

'M2CCAI2016_video107_001.png'

In [14]:
import json
import copy
from tqdm.auto import tqdm

for model in models:
    print(f"=== Using model {model} ===")
    for method in methods:
        print(f"=== Using method {method} ===")

        load_path = os.path.join(ROOT_DIR, f'results/{method}/cholec_{model}.json')
        save_path = os.path.join(ROOT_DIR, f'results/{method}/cholec_{model}_{eval_model_name}.json')

        with open(load_path) as input_file:
            results = json.load(input_file)

        new_results = []

        num_examples = len(results)
        for di in tqdm(range(num_examples)):
            result = results[di]
            
            image = test_dataset[id2idx_mapping[result['id']]]['image']
            
            example = CholecExample(
                id=result['id'],
                image = image,
                true_safe_list=result['true_safe_list'],
                true_unsafe_list=result['true_unsafe_list'],
                llm_raw_output=result['llm_raw_output'],
                llm_explanation=result['llm_explanation'],
                llm_safe_list=result['llm_safe_list'],
                llm_unsafe_list=result['llm_unsafe_list'],
            )
            example.safe_iou = result['safe_iou']
            example.unsafe_iou = result['unsafe_iou']
            
            
            # isolate individual features
            all_claims = isolate_individual_features(example.llm_explanation, model=eval_model)
            if all_claims is None:
                continue
            example.all_claims = [claim.strip() for claim in all_claims]

            # distill relevant features
            relevant_claims = distill_relevant_features(
                example.image, 
                example.all_claims,
                model=eval_model
            )
            example.relevant_claims = relevant_claims

            # calculate expert alignment scores
            align_infos = calculate_expert_alignment_scores(
                example.relevant_claims, 
                eval_model,
            )

            alignable_claims = [info["Claim"] for info in align_infos]
            alignment_categories = [info["Category"] for info in align_infos]
            aligned_category_ids = [info["Category ID"] for info in align_infos]
            alignment_scores = [info["Alignment"] for info in align_infos]
            alignment_raws = [info["Alignment Raw"] for info in align_infos]
            alignment_reasonings = [info["Reasoning"] for info in align_infos]
            
            example.alignable_claims = alignable_claims
            example.alignment_categories = alignment_categories
            example.aligned_category_ids = aligned_category_ids
            example.alignment_scores = alignment_scores
            example.alignment_raws = alignment_raws
            example.alignment_reasonings = alignment_reasonings
            
            # Non-alignable claims are given a score of 0.0
            if len(align_infos) > 0:
                example.final_alignment_score = sum(info["Alignment"] for info in align_infos) / len(example.all_claims)
            else:
                example.final_alignment_score = 0.0
            
            # save
            save_dict = {}
            for k, v in example.__dict__.items():
                if not isinstance(v, torch.Tensor):
                    save_dict[k] = v # if not isinstance(v, torch.Tensor) else v.cpu().numpy().tolist()
            # with open(save_path, 'wt') as output_file:
            #     json.dump(save_dict, output_file)

            new_results.append(save_dict)


        with open(save_path, 'wt') as output_file:
            json.dump(new_results, output_file, indent=4)

=== Using model gpt-4o ===
=== Using method vanilla ===


  0%|          | 0/150 [00:00<?, ?it/s]

=== Using model claude-3-5-sonnet-latest ===
=== Using method vanilla ===


  0%|          | 0/150 [00:00<?, ?it/s]

=== Using model gemini-2.0-flash ===
=== Using method vanilla ===


  0%|          | 0/150 [00:00<?, ?it/s]

=== Using model o1 ===
=== Using method vanilla ===


  0%|          | 0/150 [00:00<?, ?it/s]

In [15]:
import json
import copy
from tqdm.auto import tqdm

for model in models:
    print(f"=== Using model {model} ===")
    for method in methods:
        print(f"=== Using method {method} ===")

        # load_path = os.path.join(ROOT_DIR, f'results/{method}/cholec_{model}.json')
        load_path = os.path.join(ROOT_DIR, f'results/{method}/cholec_{model}_{eval_model_name}.json')
        save_path = os.path.join(ROOT_DIR, f'results/{method}/cholec_{model}_{eval_model_name}.2.json')

        with open(load_path) as input_file:
            results = json.load(input_file)

        new_results = []

        num_examples = len(results)
        for di in tqdm(range(num_examples)):
            result = results[di]
            
            image = test_dataset[id2idx_mapping[result['id']]]['image']
            
            example = CholecExample(
                id=result['id'],
                image = image,
                true_safe_list=result['true_safe_list'],
                true_unsafe_list=result['true_unsafe_list'],
                llm_raw_output=result['llm_raw_output'],
                llm_explanation=result['llm_explanation'],
                llm_safe_list=result['llm_safe_list'],
                llm_unsafe_list=result['llm_unsafe_list'],
            )
            example.safe_iou = result['safe_iou']
            example.unsafe_iou = result['unsafe_iou']
            
            
            # isolate individual features
            all_claims = isolate_individual_features(example.llm_explanation, model=eval_model)
            if all_claims is None:
                continue
            example.all_claims = [claim.strip() for claim in all_claims]

            # distill relevant features
            relevant_claims = distill_relevant_features(
                example.image, 
                example.all_claims,
                model=eval_model
            )
            example.relevant_claims = relevant_claims

            # calculate expert alignment scores
            align_infos = calculate_expert_alignment_scores(
                example.relevant_claims, 
                eval_model,
            )

            alignable_claims = [info["Claim"] for info in align_infos]
            alignment_categories = [info["Category"] for info in align_infos]
            aligned_category_ids = [info["Category ID"] for info in align_infos]
            alignment_scores = [info["Alignment"] for info in align_infos]
            alignment_raws = [info["Alignment Raw"] for info in align_infos]
            alignment_reasonings = [info["Reasoning"] for info in align_infos]
            
            example.alignable_claims = alignable_claims
            example.alignment_categories = alignment_categories
            example.aligned_category_ids = aligned_category_ids
            example.alignment_scores = alignment_scores
            example.alignment_raws = alignment_raws
            example.alignment_reasonings = alignment_reasonings
            
            # Non-alignable claims are given a score of 0.0
            if len(align_infos) > 0:
                example.final_alignment_score = sum(info["Alignment"] for info in align_infos) / len(example.all_claims)
            else:
                example.final_alignment_score = 0.0
            
            # save
            save_dict = {}
            for k, v in example.__dict__.items():
                if not isinstance(v, torch.Tensor):
                    save_dict[k] = v # if not isinstance(v, torch.Tensor) else v.cpu().numpy().tolist()
            # with open(save_path, 'wt') as output_file:
            #     json.dump(save_dict, output_file)

            new_results.append(save_dict)


        with open(save_path, 'wt') as output_file:
            json.dump(new_results, output_file, indent=4)

[]

# Cholec

# Emotion